In [37]:
# we naturally first need to import torch and torchvision
import torch
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor, Normalize, Compose
from datasets import load_dataset


def get_mnist_dataloaders(mnist_dataset, batch_size: int):
    pytorch_transforms = Compose([ToTensor(), Normalize((0.1307,), (0.3081,))])

    # Prepare transformation functions
    def apply_transforms(batch):
        batch["image"] = [pytorch_transforms(img) for img in batch["image"]]
        return batch

    mnist_train = mnist_dataset["train"].with_transform(apply_transforms)
    mnist_test = mnist_dataset["test"].with_transform(apply_transforms)

    # Construct PyTorch dataloaders
    trainloader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
    testloader = DataLoader(mnist_test, batch_size=batch_size)
    return trainloader, testloader


# Download dataset
mnist = load_dataset("ylecun/mnist")

In [38]:
import torch.nn as nn
import torch.nn.functional as F
class Net(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [39]:
def train(net, trainloader, optimizer, device="cpu"):
    """Train the network on the training set."""
    criterion = torch.nn.CrossEntropyLoss()
    net.to(device)
    net.train()
    for batch in trainloader:
        images, labels = batch["image"].to(device), batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(net(images), labels)
        loss.backward()
        optimizer.step()


def test(net, testloader, device):
    """Validate the network on the entire test set."""
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    net.to(device)
    net.eval()
    with torch.no_grad():
        for batch in testloader:
            images, labels = batch["image"].to(device), batch["label"].to(device)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
    accuracy = correct / len(testloader.dataset)
    return loss, accuracy

def run_centralised(
    trainloader, testloader, epochs: int, lr: float, momentum: float = 0.9
):
    """A minimal (but complete) training loop"""

    # instantiate the model
    model = Net(num_classes=10)

    # Discover device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # define optimiser with hyperparameters supplied
    optim = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    # train for the specified number of epochs
    for e in range(epochs):
        print(f"Training epoch {e} ...")
        train(model, trainloader, optim, device)

    # training is completed, then evaluate model on the test set
    loss, accuracy = test(model, testloader, device)
    print(f"{loss = }")
    print(f"{accuracy = }")

In [49]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from flwr_datasets.partitioner import DirichletPartitioner
from flwr_datasets.partitioner import PathologicalPartitioner

NUM_PARTITIONS = 100

# partitioner = IidPartitioner(num_partitions=NUM_PARTITIONS)
# Let's partition the "train" split of the MNIST dataset
# The MNIST dataset will be downloaded if it hasn't been already
# fds = FederatedDataset(dataset="ylecun/mnist", partitioners={"train": partitioner})


# dirichlet_partitioner = DirichletPartitioner(
#     num_partitions=10, alpha=0.2, partition_by="label"
# )
# fds = FederatedDataset(dataset="ylecun/mnist", partitioners={"train":dirichlet_partitioner})


pathological_partitioner = PathologicalPartitioner(
    num_partitions=10, partition_by="label", num_classes_per_partition=2
)
fds = FederatedDataset(dataset="ylecun/mnist", partitioners={"train":pathological_partitioner})

In [50]:
import tenseal as ts
import flwr as fl
import random
import numpy as np
poly_modulus_degree= 8192
def get_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=poly_modulus_degree,
        coeff_mod_bit_sizes=[60, 40, 60]
    )
    context.generate_galois_keys()
    context.global_scale = 2**40
    return context

shared_context = get_context()
server_context = shared_context.serialize(save_secret_key=False)
client_context = shared_context.serialize(save_secret_key=True)

chunk_size = poly_modulus_degree//2
# use_he = False
use_he = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # For cudnn backend — makes things deterministic 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



In [51]:
def encrypt_params(params, context, poly_modulus_degree):
    """
    Encrypts model parameters using TenSEAL CKKS vectors.
    
    Args:
        params (List[np.ndarray]): List of model parameters (numpy arrays).
        context (ts.Context): TenSEAL context.
        poly_modulus_degree (int): Used to determine chunk size.

    Returns:
        List[List[ts.CKKSVector]]: Encrypted chunks for each parameter.
    """
    enc_params = []
    chunk_size = poly_modulus_degree // 2

    for p in params:
        flat = p.flatten()
        chunks = torch.split(torch.from_numpy(flat), chunk_size)

        encrypted_chunks = []
        for i, chunk in enumerate(chunks):
            # print(f"Chunk {i + 1}: length = {len(chunk)}")  # 👈 Add this line
            chunk_list = chunk.tolist()
            enc_vec = ts.ckks_vector(context, chunk_list)
            serialized_vec = enc_vec.serialize()
            encrypted_chunks.append(serialized_vec)

        enc_params.append(encrypted_chunks)

    return enc_params
import numpy as np
import tenseal as ts

def decrypt_params(flat_enc_list, context, model, chunk_size):
    decrypted_params = []
    state_dict_keys = list(model.state_dict().keys())
    
    idx = 0  # index in flat_enc_list
    
    for param_idx, key in enumerate(state_dict_keys):
        target_shape = model.state_dict()[key].shape
        param_size = np.prod(target_shape)  # total elements in this param
        
        # Calculate how many chunks this param should have based on chunk_size
        num_chunks = (param_size + chunk_size - 1) // chunk_size  # ceiling division
        
        enc_chunks = flat_enc_list[idx : idx + num_chunks]
        idx += num_chunks
        
        decrypted_chunks = []
        for serialized_vec in enc_chunks:
            ckks_vec = ts.ckks_vector_from(context, serialized_vec)  # Deserialize
            decrypted_chunks.extend(ckks_vec.decrypt())
        
        # Trim to the exact param size (in case last chunk was padded)
        decrypted_chunks = decrypted_chunks[:param_size]
        
        decrypted_array = np.array(decrypted_chunks, dtype=np.float32).reshape(target_shape)
        decrypted_params.append(decrypted_array)
    
    return decrypted_params

In [52]:
from collections import OrderedDict
from typing import Dict, Tuple

import torch 
from flwr.common import NDArrays, Scalar
from flwr.client import NumPyClient
import sys
import time
import os
class FlowerClient(fl.client.NumPyClient):
    set_seed(42)
    def __init__(self, trainloader, valloader,context) -> None:
        super().__init__()

        self.trainloader = trainloader
        self.valloader = valloader
        self.model = Net(num_classes=10)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.context = context
        self.last_fit_total_time = 0
        
    def fit(self, parameters, config):
        print("==This is fit method==")
        start_total = time.time()
        server_round = config.get("server_round", -1)
        if use_he:
            # === HE mode ===
            start_decrypt = time.time()
            enc_param_list = [bytes(ndarray) for ndarray in parameters]
            chunked_enc_params = [enc_param_list]
            decrypted_params = decrypt_params(enc_param_list, self.context, self.model, chunk_size)
            end_decrypt = time.time()
            # print("== Decrypted parameter data types ==")
            # for i, param in enumerate(decrypted_params):
            #     print(f"Param {i}: type={type(param)}, dtype={param.dtype}, shape={param.shape}")
            
            for param_tensor, decrypted_np in zip(self.model.parameters(), decrypted_params):
                param_tensor.data = torch.from_numpy(decrypted_np).to(self.device)
        else:
            # === Plaintext mode ===
            set_params(self.model, parameters)
        #  measure on global before training
        loss_global, acc_global = test(self.model, self.valloader, self.device)
        start_train = time.time()
        optim = torch.optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)
        train(self.model, self.trainloader, optim, self.device)
        # measure on local model
        loss_personalized, acc_personalized = test(self.model, self.valloader, self.device)
        end_train = time.time()

        updated_params = [param_tensor.detach().cpu().numpy() for param_tensor in self.model.parameters()]

        if use_he:
            start_encrypt = time.time()
            encrypted_chunks = encrypt_params(updated_params, self.context, poly_modulus_degree)
            encrypted_bytes_list = [chunk for param in encrypted_chunks for chunk in param]
            end_encrypt = time.time()

            sent_bytes = sum(len(chunk) for chunk in encrypted_bytes_list)
            end_total = time.time()
            print("===Bandwidth usage===")
            print("Bandwidth with HE: ", sent_bytes)
            print("===Runtime performance===")
            print(f"⏱️ Decryption time: {end_decrypt - start_decrypt:.3f} s")
            print(f"⏱️ Training time: {end_train - start_train:.3f} s")
            print(f"⏱️ Encryption time: {end_encrypt - start_encrypt:.3f} s")
            print(f"⏱️ Total fit() time: {end_total - start_total:.3f} s")
            self.last_fit_total_time = end_total - start_total

            return encrypted_bytes_list, len(self.trainloader),{
                "fit_total_time": self.last_fit_total_time,
                "bandwidth": sent_bytes,
                "loss on local model": loss_personalized,
                "accuracy on personalized model": acc_personalized,
                "loss on global model before traning": loss_global,
                "accuracy on global model": acc_global
            }
        else:
            sent_bytes = sum(p.nbytes for p in updated_params)
            end_total = time.time()
            print("===Bandwidth usage===")
            print("Bandwidth without HE: ", sent_bytes)
            print("===Runtime performance===")
            print(f"⏱️ Training time: {end_train - start_train:.3f} s")
            print(f"⏱️ Total fit() time: {end_total - start_total:.3f} s")
            self.last_fit_total_time = end_total - start_total

            return get_params(self.model), len(self.trainloader),{
                "fit_total_time": self.last_fit_total_time,
                "bandwidth": sent_bytes,
            }

    def evaluate(self, parameters: NDArrays, config: Dict[str, Scalar]):
        print("==This is evaluate method==")
        server_round = config.get("server_round", -1) 
        print(f"== Evaluation at Round {server_round} ==")
        start_eval = time.time()
        if use_he:
            flat_enc_params = [bytes(ndarray) for ndarray in parameters]
            received_bytes = sum(len(chunk) for chunk in flat_enc_params)
            decrypted_params = decrypt_params(flat_enc_params, self.context, self.model,chunk_size=poly_modulus_degree // 2)
            for param_tensor, decrypted_np in zip(self.model.parameters(), decrypted_params):
                param_tensor.data = torch.from_numpy(decrypted_np).to(self.device)
        else:
            set_params(self.model, parameters)
            received_bytes = sum(p.nbytes for p in parameters)
            
        loss, accuracy = test(self.model, self.valloader, self.device)
        end_eval = time.time()
        print(f"⏱️ Total evaluate() time: {end_eval - start_eval:.3f} s")
        print(f"===Bandwidth usage===")
        print(f"Bandwidth (received): {received_bytes} bytes")

        metrics = {
            "accuracy": float(accuracy),
            "eval_total_time": end_eval - start_eval,
            "bandwidth": received_bytes,
            "fit_total_time": self.last_fit_total_time,
        }
        # if self.last_fit_total_time is not None:
        #     metrics["fit_total_time"] = self.last_fit_total_time
    
        return float(loss), len(self.valloader), metrics #{"accuracy": float(accuracy)}

# # Two auxhiliary functions to set and extract parameters of a model
def set_params(model, parameters):
    """Replace model parameters with those passed as `parameters`."""

    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.from_numpy(v) for k, v in params_dict})
    # now replace the parameters
    model.load_state_dict(state_dict, strict=True)

def get_params(model):
    """Extract model parameters as a list of NumPy arrays."""
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

In [53]:
from flwr.common import Context
from flwr.client import ClientApp
   

def client_fn(context: Context) -> fl.client.Client:
    """Returns a FlowerClient containing its data partition."""

    partition_id = int(context.node_config["partition-id"])
    partition = fds.load_partition(partition_id, "train")
    # partition into train/validation
    partition_train_val = partition.train_test_split(test_size=0.1, seed=42)

    # Let's use the function defined earlier to construct the dataloaders
    # and apply the dataset transformations
    trainloader, testloader = get_mnist_dataloaders(partition_train_val, batch_size=32)
    context = ts.context_from(client_context)

    return FlowerClient(trainloader=trainloader, valloader=testloader,context=context).to_client()


# Concstruct the ClientApp passing the client generation function
client_app = ClientApp(client_fn=client_fn)

In [54]:
from typing import List
from flwr.common import Metrics


# Define metric aggregation function
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metric (weighted average)
    return {"accuracy": sum(accuracies) / sum(examples)}

In [55]:
import flwr as fl
from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy


def ndarrays_to_byteparam(ckks_vectors: list[ts.CKKSVector]) -> fl.common.Parameters:
    """
    Convert a list of CKKSVector to serialized Flower Parameters.
    """
    serialized = [vec.serialize() for vec in ckks_vectors]
    return fl.common.ndarrays_to_parameters(serialized)

class BytesStrategy(fl.server.strategy.FedAvg):
    def __init__(self, context: fl.common.Context, **kwargs):
        super().__init__(**kwargs)
        self.context = context
    
    
    def aggregate_fit(self, server_round, results, failures):
        # print(f"\n[Round {server_round}] Starting encrypted aggregation")
        print("Starting encrypted aggregation")
        
        if failures:
            print(f"⚠️  {len(failures)} client(s) failed.")
            for i, failure in enumerate(failures):
                print(f"  - Failure {i+1}: {repr(failure)}")

        # Step 1: Deserialize all encrypted chunks for all clients
        all_encrypted = []
        for client, fit_res in results:
            client_chunks = []
            for serialized_chunk in fl.common.parameters_to_ndarrays(fit_res.parameters):
                ckks_vec = ts.ckks_vector_from(self.context, bytes(serialized_chunk))
                client_chunks.append(ckks_vec)
            all_encrypted.append(client_chunks)

        if not all_encrypted:
            print("❌ No valid client updates received. Skipping round.")
            return None, {}

        # Step 2: Aggregate corresponding chunks across clients
        num_chunks = len(all_encrypted[0])
        aggregated_chunks = []

        for idx in range(num_chunks):
            chunk_group = [client_chunks[idx] for client_chunks in all_encrypted]
            avg_chunk = chunk_group[0]
            for vec in chunk_group[1:]:
                avg_chunk += vec
            avg_chunk *= 1 / len(chunk_group)
            aggregated_chunks.append(avg_chunk)
        fit_times = []
        for client, fit_res in results:
            fit_time = fit_res.metrics.get("fit_total_time", 0.0)
            fit_times.append(fit_time)

        avg_fit_time = sum(fit_times) / len(fit_times) if fit_times else 0.0
        # Collect personalized loss and accuracy from each client
        losses = [fit_res.metrics.get("loss on local model", 0.0) for _, fit_res in results]
        accuracies = [fit_res.metrics.get("accuracy on personalized model", 0.0) for _, fit_res in results]

        glo_loss = [fit_res.metrics.get("loss on global model before traning", 0.0) for _, fit_res in results]
        glo_accuracies = [fit_res.metrics.get("accuracy on global model", 0.0) for _, fit_res in results]

        # Calculate averages (you can also do weighted average if you want, using client data size)
        avg_loss = sum(losses) / len(losses) if losses else 0.0
        avg_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0.0

        g_loss = sum(glo_loss) / len(glo_loss) if losses else 0.0
        g_acc = sum(glo_accuracies) / len(glo_accuracies) if losses else 0.0

        print(f"⏱️ Round {server_round} average personalized loss: {avg_loss:.4f}")
        print(f"⏱️ Round {server_round} average personalized accuracy: {avg_accuracy:.4f}")
        print(f"⏱️ Round {server_round} average fit time: {avg_fit_time:.3f} s")

        # Step 4: Return aggregated result as Flower Parameters
        new_params = ndarrays_to_byteparam(aggregated_chunks)
        serialized = vec.serialize()
        # print("[SERVER DEBUG] First 100 bytes of chunk 0:", serialized[:100])

        return new_params,  {
        "fit_total_time": avg_fit_time,
        "loss value on personalized model": avg_loss,
        "avg_accuracy on personalized model": avg_accuracy,
        "loss value before local traning:": g_loss,
        "accuracy on global model before traing": g_acc
    }
    
    def configure_evaluate(
        self,
        server_round: int,
        parameters: fl.common.Parameters,
        client_manager: fl.server.client_manager.ClientManager,
    ) -> List[Tuple[fl.server.client_proxy.ClientProxy, fl.common.EvaluateIns]]:
        
        # ✅ Call the parent method to get the client instructions
        client_instructions = super().configure_evaluate(server_round, parameters, client_manager)

        # ✅ Inject server_round into config
        updated_instructions = []
        for client, evaluate_ins in client_instructions:
            new_config = dict(evaluate_ins.config)
            new_config["server_round"] = server_round  # 👈 Add round here
            updated_ins = fl.common.EvaluateIns(evaluate_ins.parameters, new_config)
            updated_instructions.append((client, updated_ins))

        return updated_instructions
    
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:

        if not results:
            return None, {}
        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        # Collect values
        examples = [r.num_examples for _, r in results]
        total_examples = sum(examples)

        accuracies = [r.metrics["accuracy"] * r.num_examples for _, r in results]
        fit_times = [r.metrics["fit_total_time"] for _, r in results]
        # print("Fit times: ", fit_times)
        # print("Accuracies: ", accuracies)
        eval_times = [r.metrics.get("eval_total_time", 0.0) for _, r in results]
        bandwidths = [r.metrics.get("bandwidth", 0.0) for _, r in results]

        # Aggregate
        aggregated_accuracy = sum(accuracies) / total_examples
        # avg_fit_time = sum(fit_times) / len(fit_times) if fit_times else 0.0
        avg_eval_time = sum(eval_times) / len(eval_times) if eval_times else 0.0
        avg_bandwidth = sum(bandwidths) / len(bandwidths) if bandwidths else 0.0


        return float(aggregated_loss), {
            "accuracy": float(aggregated_accuracy),
            #"fit_total_time": avg_fit_time
            "eval_total_time": avg_eval_time,
            "bandwidth": avg_bandwidth,
        }

In [56]:
from flwr.common import ndarrays_to_parameters
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
import random

num_rounds = 5

def server_fn(context: Context):
    set_seed(42)
    model = Net(num_classes=10)
    plain_params = get_params(model)

    if use_he:
        ckks_chunks = encrypt_params(plain_params, shared_context, poly_modulus_degree)
        flat_encrypted = [chunk for param in ckks_chunks for chunk in param]
        global_model_init = ndarrays_to_parameters(flat_encrypted)

        strategy = BytesStrategy(
            context=shared_context,
            fraction_fit=1,
            fraction_evaluate=1,
            initial_parameters=global_model_init,
            evaluate_metrics_aggregation_fn=weighted_average,
        )
    else:
        global_model_init = ndarrays_to_parameters(plain_params)

        strategy = FedAvg(
            fraction_fit=1,
            fraction_evaluate=1,
            initial_parameters=global_model_init,
            evaluate_metrics_aggregation_fn=weighted_average,
        )

    config = ServerConfig(num_rounds=num_rounds)
    return ServerAppComponents(strategy=strategy, config=config)

# Create your ServerApp
server_app = ServerApp(server_fn=server_fn)


In [57]:
from flwr.simulation import run_simulation

num_clients = 3
run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=num_clients, backend_config={"num_cpus": 1},
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
(raylet) /Users/creamy/Research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
(raylet)   warnings.warn(
(ClientAppActor pid=24701) /Users/creamy/Research/.venv/lib/python3.9/site-packages/flwr_datasets/partitioner/pathological_partitioner.py:188: UserWarning: Classes: [2, 8, 9] will NOT be used due to the chosen configuration. If it is undesired behavior consider setting 'first_class_deterministic_assignment=True' which in

(ClientAppActor pid=24701) ==This is fit method==
(ClientAppActor pid=24701) ===Bandwidth usage===
(ClientAppActor pid=24701) Bandwidth with HE:  4472091
(ClientAppActor pid=24701) ===Runtime performance===
(ClientAppActor pid=24701) ⏱️ Decryption time: 0.020 s
(ClientAppActor pid=24701) ⏱️ Training time: 0.537 s
(ClientAppActor pid=24701) ⏱️ Encryption time: 0.050 s
(ClientAppActor pid=24701) ⏱️ Total fit() time: 0.660 s


INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 clients (out of 3)


Starting encrypted aggregation
⏱️ Round 1 average personalized loss: 0.4852
⏱️ Round 1 average personalized accuracy: 0.9919
⏱️ Round 1 average fit time: 0.932 s


(ClientAppActor pid=24699) /Users/creamy/Research/.venv/lib/python3.9/site-packages/flwr_datasets/partitioner/pathological_partitioner.py:188: UserWarning: Classes: [2, 8, 9] will NOT be used due to the chosen configuration. If it is undesired behavior consider setting 'first_class_deterministic_assignment=True' which in case when the number of classes is smaller than the number of partitions will utilize all the classes for the created partitions. [repeated 3x across cluster]
(ClientAppActor pid=24699)   warnings.warn( [repeated 3x across cluster]


...

(ClientAppActor pid=24699) /Users/creamy/Research/.venv/lib/python3.9/site-packages/flwr_datasets/partitioner/pathological_partitioner.py:188: UserWarning: Classes: [2, 8, 9] will NOT be used due to the chosen configuration. If it is undesired behavior consider setting 'first_class_deterministic_assignment=True' which in case when the number of classes is smaller than the number of partitions will utilize all the classes for the created partitions. [repeated 3x across cluster]
(ClientAppActor pid=24699)   warnings.warn( [repeated 3x across cluster]


(ClientAppActor pid=24699) ==This is evaluate method==
(ClientAppActor pid=24699) == Evaluation at Round 5 ==
(ClientAppActor pid=24699) ==This is fit method== [repeated 2x across cluster]


INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 63.07s
INFO :      	History (loss, distributed):
INFO :      		round 1: 18.91484754851886
INFO :      		round 2: 10.654524473916917
INFO :      		round 3: 3.516541428154423
INFO :      		round 4: 6.019623228526187
INFO :      		round 5: 5.691524587926411
INFO :      	History (metrics, distributed, fit):
INFO :      	{'accuracy on global model before traing': [(1, 0.38751602705172733),
INFO :      	                                            (2, 0.5339199006848768),
INFO :      	                                            (3, 0.758423221273074),
INFO :      	                                            (4, 0.9362685725452294),
INFO :      	                                            (5, 0.9095718999256079)],
INFO :      	 'avg_accuracy on personalized model': [(1, 0.9919127374528566),
INFO :      	                                        (2, 0.9938695

(ClientAppActor pid=24700) ⏱️ Total evaluate() time: 0.049 s [repeated 3x across cluster]
(ClientAppActor pid=24700) Bandwidth (received): 2493118 bytes [repeated 3x across cluster]
(ClientAppActor pid=24700) ===Bandwidth usage=== [repeated 6x across cluster]
(ClientAppActor pid=24699) Bandwidth with HE:  4472213 [repeated 2x across cluster]
(ClientAppActor pid=24699) ===Runtime performance=== [repeated 2x across cluster]
(ClientAppActor pid=24699) ⏱️ Decryption time: 0.009 s [repeated 2x across cluster]
(ClientAppActor pid=24699) ⏱️ Training time: 0.777 s [repeated 2x across cluster]
(ClientAppActor pid=24699) ⏱️ Encryption time: 0.047 s [repeated 2x across cluster]
(ClientAppActor pid=24699) ⏱️ Total fit() time: 0.889 s [repeated 2x across cluster]
(ClientAppActor pid=24700) ==This is evaluate method== [repeated 2x across cluster]
(ClientAppActor pid=24700) == Evaluation at Round 5 == [repeated 2x across cluster]


(ClientAppActor pid=24700) /Users/creamy/Research/.venv/lib/python3.9/site-packages/flwr_datasets/partitioner/pathological_partitioner.py:188: UserWarning: Classes: [2, 8, 9] will NOT be used due to the chosen configuration. If it is undesired behavior consider setting 'first_class_deterministic_assignment=True' which in case when the number of classes is smaller than the number of partitions will utilize all the classes for the created partitions. [repeated 2x across cluster]
(ClientAppActor pid=24700)   warnings.warn( [repeated 2x across cluster]
